In [2]:
import requests
import pandas as pd
from typing import Dict, List, Optional

class INECensusAPI:
    """Corrected client for INE OpenAPI"""
    
    # Base URLs for different INE API services
    TEMPUS_API = "https://servicios.ine.es/wstempus/js/EN"
    PC_AXIS_API = "https://www.ine.es/jaxiT3/DatosJson.do"
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'Accept': 'application/json',
            'User-Agent': 'INE-API-Client/1.0'
        })
    
    def get_operations(self) -> List[Dict]:
        """Get all statistical operations including census operations"""
        url = f"{self.TEMPUS_API}/operaciones"
        
        response = self.session.get(url)
        response.raise_for_status()
        return response.json()
    
    def get_tables_by_operation(self, operation_id: int) -> List[Dict]:
        """Get tables for a specific operation"""
        url = f"{self.TEMPUS_API}/operacion/{operation_id}/tablas"
        
        response = self.session.get(url)
        response.raise_for_status()
        return response.json()
    
    def get_table_data(self, table_id: int) -> Dict:
        """Get data from a specific table"""
        url = f"{self.TEMPUS_API}/tabla/{table_id}"
        
        response = self.session.get(url)
        response.raise_for_status()
        return response.json()
    
    def get_series_data(self, series_id: str) -> Dict:
        """Get data from a specific series"""
        url = f"{self.TEMPUS_API}/serie/{series_id}"
        
        response = self.session.get(url)
        response.raise_for_status()
        return response.json()
    
    def search_census_data(self, search_term: str = "censo") -> List[Dict]:
        """Search for census-related data"""
        # Get all operations and filter for census
        operations = self.get_operations()
        
        census_ops = []
        for op in operations:
            op_name = op.get('Nombre', '').lower()
            if search_term.lower() in op_name:
                census_ops.append(op)
                print(f"Found: {op.get('Nombre')} (ID: {op.get('Id')})")
        
        return census_ops

# ============================================================================
# WORKING EXAMPLE FOR CENSUS DATA
# ============================================================================

def get_census_population_data():
    """Get population census data using working INE API endpoints"""
    
    api = INECensusAPI()
    
    print("=" * 60)
    print("FETCHING SPANISH CENSUS DATA")
    print("=" * 60)
    
    # Step 1: Find census operations
    print("\n📊 Step 1: Finding census operations...")
    
    try:
        # Get all operations
        all_ops = api.get_operations()
        
        # Filter for census operations
        census_operations = []
        for op in all_ops:
            name = op.get('Nombre', '').lower()
            if any(keyword in name for keyword in ['censo', 'población', 'population', 'housing']):
                census_operations.append(op)
                print(f"  - {op.get('Id')}: {op.get('Nombre')}")
        
        if not census_operations:
            print("  No census operations found. Here are some available operations:")
            for op in all_ops[:10]:  # Show first 10
                print(f"  - {op.get('Id')}: {op.get('Nombre')}")
            return None
        
        # Step 2: Get tables from the first census operation
        census_op = census_operations[0]
        op_id = census_op['Id']
        op_name = census_op['Nombre']
        
        print(f"\n📋 Step 2: Getting tables from '{op_name}' (ID: {op_id})...")
        
        tables = api.get_tables_by_operation(op_id)
        
        if tables:
            print(f"  Found {len(tables)} tables")
            for table in tables[:5]:  # Show first 5
                print(f"    - Table {table.get('Id')}: {table.get('Nombre', 'Unknown')}")
            
            # Step 3: Get data from the first table
            first_table = tables[0]
            table_id = first_table['Id']
            
            print(f"\n📈 Step 3: Fetching data from table {table_id}...")
            
            try:
                table_data = api.get_table_data(table_id)
                
                # Extract actual data values if present
                if 'Data' in table_data:
                    data = table_data['Data']
                    print(f"  Retrieved {len(data)} data points")
                    
                    # Convert to DataFrame
                    df = pd.DataFrame(data)
                    print("\n  First 10 rows of census data:")
                    print(df.head(10))
                    
                    return df
                else:
                    print(f"  Table structure: {list(table_data.keys())}")
                    return table_data
                    
            except Exception as e:
                print(f"  Error fetching table data: {e}")
        else:
            print("  No tables found for this operation")
            
    except requests.exceptions.HTTPError as e:
        print(f"  HTTP Error: {e}")
        print("  Trying alternative API endpoint...")
        
        # Alternative approach using PC-Axis API
        return get_census_data_alternative()

def get_census_data_alternative():
    """Alternative method using PC-Axis API"""
    
    print("\n🔄 Using PC-Axis API alternative...")
    
    # Known census table IDs (you'll need to find the correct ones)
    # These are examples - replace with actual census table IDs
    census_tables = [
        {"id": "27186", "name": "Population by municipality"},  # Example
        {"id": "27187", "name": "Population by age groups"}
    ]
    
    for table in census_tables:
        try:
            url = f"https://www.ine.es/jaxiT3/DatosJson.do?c={table['id']}"
            response = requests.get(url)
            
            if response.status_code == 200:
                data = response.json()
                print(f"\n✅ Retrieved data from table {table['id']}: {table['name']}")
                
                # Process the data
                if isinstance(data, list) and len(data) > 0:
                    df = pd.DataFrame(data)
                    print(f"  Shape: {df.shape}")
                    print(df.head())
                    return df
                    
        except Exception as e:
            print(f"  Could not fetch table {table['id']}: {e}")
    
    return None

def get_population_by_municipality():
    """Specific function to get population by municipality"""
    
    # This endpoint might work for municipality-level data
    # You need the correct table ID from INE
    
    print("\n🏙️ Fetching population by municipality...")
    
    # Common census table IDs (verify these on INE website first)
    # You can find these by browsing INEbase and checking the table URL
    
    municipality_endpoints = [
        "https://www.ine.es/jaxiT3/DatosJson.do?c=27186",  # Example - replace!
        "https://servicios.ine.es/wstempus/js/EN/tabla/27186"
    ]
    
    for endpoint in municipality_endpoints:
        try:
            response = requests.get(endpoint)
            if response.status_code == 200:
                data = response.json()
                print(f"  Success from: {endpoint}")
                
                if isinstance(data, list):
                    df = pd.DataFrame(data)
                    print(f"  Retrieved {len(df)} rows")
                    return df
                    
        except Exception as e:
            continue
    
    print("  Could not retrieve municipality data. You need to find the correct table ID.")
    print("\n  To find census table IDs:")
    print("  1. Go to https://www.ine.es/dyngs/INEbase/en/operacion.htm?c=Estadistica_C&cid=1254736177000")
    print("  2. Navigate to 'Population and Housing Census 2021'")
    print("  3. Find a table you want")
    print("  4. Look for the 'id' parameter in the URL or use browser dev tools")
    
    return None

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    
    # Test the corrected API
    print("Testing INE Census API...\n")
    
    # Method 1: Use the TEMPUS API
    try:
        census_data = get_census_population_data()
        if census_data is not None:
            print("\n✅ Successfully retrieved census data!")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        
        # Method 2: Try the alternative approach
        print("\nAttempting alternative approach...")
        alt_data = get_census_data_alternative()
        
        # Method 3: Try municipality-level data
        if alt_data is None:
            mun_data = get_population_by_municipality()

Testing INE Census API...

FETCHING SPANISH CENSUS DATA

📊 Step 1: Finding census operations...
  - 8: Population and Housing Censuses  
  - 15: Housing Price Index (HPI) 
  - 21: Population Now Cast (ePOBa) 
  - 22: Official population figures of the Spanish Municipalities: Revision of the Municipal Register
  - 29: Economically Active Population Survey (EAPS) 
  - 35: DPOH OPERATION  De facto Population figures from 1900 until 1991
  - 36: Dejure Population figures from 1986 until 1995
  - 52: Short-Term Population Projections 
  - 54: Long-Term Population Projections 
  - 72: Population Figures
  - 205: Economically Active Population Flows  
  - 293: Economically Active Population Survey
  - 326: Population Projections
  - 378: General Statistics on the Inmate Population. Monthly Periodicity
  - 428: Exploitation of Educational Variables in the Economically Active Population Survey: Level of Training and On-going Training
  - 432: Rental Housing Price Index.
  - 440: Exploitation of

In [3]:
import requests
import pandas as pd
from typing import Dict, List, Optional

class INECensusAPI:
    """Working client for INE Census API"""
    
    TEMPUS_API = "https://servicios.ine.es/wstempus/js/EN"
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'Accept': 'application/json',
            'User-Agent': 'INE-API-Client/1.0'
        })
    
    def get_operations(self) -> List[Dict]:
        """Get all statistical operations"""
        url = f"{self.TEMPUS_API}/operaciones"
        response = self.session.get(url)
        response.raise_for_status()
        return response.json()
    
    def get_tables_by_operation(self, operation_id: int) -> List[Dict]:
        """Get tables for a specific operation"""
        url = f"{self.TEMPUS_API}/operacion/{operation_id}/tablas"
        response = self.session.get(url)
        response.raise_for_status()
        return response.json()
    
    def get_table_data(self, table_id: int) -> Dict:
        """Get data from a specific table"""
        url = f"{self.TEMPUS_API}/tabla/{table_id}"
        response = self.session.get(url)
        response.raise_for_status()
        return response.json()
    
    def get_table_structure(self, table_id: int) -> Dict:
        """Get metadata about a table without the full data"""
        url = f"{self.TEMPUS_API}/tabla/{table_id}/meta"
        response = self.session.get(url)
        response.raise_for_status()
        return response.json()

def explore_census_tables():
    """Explore available census tables and their structure"""
    
    api = INECensusAPI()
    
    print("=" * 70)
    print("EXPLORING SPANISH CENSUS DATA")
    print("=" * 70)
    
    # Step 1: Get the Population and Housing Censuses operation
    print("\n📊 Step 1: Finding census operations...")
    operations = api.get_operations()
    
    census_op = None
    for op in operations:
        if op.get('Id') == 8:  # Population and Housing Censuses
            census_op = op
            print(f"  ✓ Found: {op.get('Nombre')} (ID: {op.get('Id')})")
            break
    
    if not census_op:
        print("  Census operation not found")
        return None
    
    # Step 2: Get all tables from this operation
    print(f"\n📋 Step 2: Getting tables from operation {census_op['Id']}...")
    tables = api.get_tables_by_operation(8)  # ID 8 for Population and Housing Censuses
    
    print(f"  Found {len(tables)} tables:\n")
    
    # Display table information
    table_info = []
    for i, table in enumerate(tables, 1):
        table_id = table.get('Id')
        table_name = table.get('Nombre', 'Unknown')
        # Get first few words of description
        desc = table.get('Descripcion', '')[:100] if table.get('Descripcion') else 'No description'
        
        table_info.append({
            'index': i,
            'id': table_id,
            'name': table_name,
            'description': desc
        })
        
        print(f"  Table {i}:")
        print(f"    ID: {table_id}")
        print(f"    Name: {table_name}")
        print(f"    Description: {desc}...")
        print()
    
    return table_info

def fetch_census_table_data(table_id: int, max_rows: int = 100):
    """Fetch and display data from a specific census table
    
    Args:
        table_id: The table ID to fetch (from explore_census_tables)
        max_rows: Maximum number of rows to display
    """
    
    api = INECensusAPI()
    
    print(f"\n📈 Fetching data from table {table_id}...")
    
    try:
        # Get table data
        table_data = api.get_table_data(table_id)
        
        print(f"\n  Table structure keys: {list(table_data.keys())}")
        
        # The actual data is usually in 'Data' or similar field
        if 'Data' in table_data:
            data = table_data['Data']
            print(f"  Found {len(data)} data points")
            
            # Convert to DataFrame
            if isinstance(data, list) and len(data) > 0:
                df = pd.DataFrame(data)
                print(f"\n  Data shape: {df.shape}")
                print(f"  Columns: {list(df.columns)}")
                
                print(f"\n  First {min(max_rows, len(df))} rows:")
                print(df.head(max_rows))
                
                return df
            else:
                print(f"  Data format: {type(data)}")
                return data
        else:
            # If no 'Data' field, show the entire structure
            print(f"\n  Table metadata:")
            for key, value in table_data.items():
                if key != 'Data' and key != 'Series':
                    print(f"    {key}: {str(value)[:100]}...")
            
            # Check for Series data
            if 'Series' in table_data:
                series = table_data['Series']
                print(f"\n  Found {len(series)} series")
                for s in series[:3]:  # Show first 3 series
                    print(f"    Series ID: {s.get('Id')} - {s.get('Nombre', 'Unknown')}")
            
            return table_data
            
    except Exception as e:
        print(f"  Error fetching table {table_id}: {e}")
        return None

def get_series_data(series_id: str):
    """Fetch data from a specific series"""
    
    api = INECensusAPI()
    
    print(f"\n📈 Fetching series {series_id}...")
    
    try:
        series_data = api.get_series_data(series_id)
        
        if 'Data' in series_data:
            data = series_data['Data']
            df = pd.DataFrame(data)
            print(f"  Retrieved {len(df)} observations")
            print(df.head())
            return df
        else:
            print(f"  Series structure: {list(series_data.keys())}")
            return series_data
            
    except Exception as e:
        print(f"  Error: {e}")
        return None

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    
    # Step 1: Explore available census tables
    print("Step 1: Exploring census tables...")
    tables = explore_census_tables()
    
    if tables:
        print("\n" + "=" * 70)
        print("Step 2: Fetching data from the first table")
        print("=" * 70)
        
        # Get the first table ID from the list
        first_table_id = tables[0]['id']
        print(f"\nFetching table {first_table_id}: {tables[0]['name']}")
        
        # Fetch data from the first table
        census_df = fetch_census_table_data(first_table_id)
        
        # If you want to try other tables, uncomment and modify:
        # second_table_id = tables[1]['id']
        # census_df2 = fetch_census_table_data(second_table_id)
        
    else:
        print("\n❌ No tables found. Trying alternative approach...")
        
        # Alternative: Try known census table IDs
        # These IDs are examples - you'll find real ones from the exploration above
        test_ids = [27186, 27187, 27188]  # Replace with actual IDs found
        
        for test_id in test_ids:
            print(f"\nTrying table {test_id}...")
            result = fetch_census_table_data(test_id)
            if result is not None:
                break

Step 1: Exploring census tables...
EXPLORING SPANISH CENSUS DATA

📊 Step 1: Finding census operations...
  ✓ Found: Population and Housing Censuses   (ID: 8)

📋 Step 2: Getting tables from operation 8...
  Found 5 tables:



AttributeError: 'str' object has no attribute 'get'

In [4]:
import requests
import pandas as pd
import json
from typing import Dict, List, Optional, Any

class INECensusAPI:
    """Working client for INE Census API - handles actual response formats"""
    
    TEMPUS_API = "https://servicios.ine.es/wstempus/js/EN"
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'Accept': 'application/json',
            'User-Agent': 'INE-API-Client/1.0'
        })
    
    def get_operations(self) -> List[Dict]:
        """Get all statistical operations"""
        url = f"{self.TEMPUS_API}/operaciones"
        response = self.session.get(url)
        response.raise_for_status()
        data = response.json()
        # Handle both list and dict responses
        return data if isinstance(data, list) else data.get('operaciones', [])
    
    def get_tables_by_operation(self, operation_id: int) -> List[Dict]:
        """Get tables for a specific operation"""
        url = f"{self.TEMPUS_API}/operacion/{operation_id}/tablas"
        response = self.session.get(url)
        response.raise_for_status()
        data = response.json()
        
        # Handle different response formats
        if isinstance(data, list):
            return data
        elif isinstance(data, dict):
            # Check for common wrapper keys
            for key in ['tablas', 'tables', 'data', 'resultados']:
                if key in data:
                    return data[key]
            return [data]  # Single table wrapped in dict
        else:
            return []

def explore_census_tables():
    """Explore available census tables and their structure"""
    
    api = INECensusAPI()
    
    print("=" * 70)
    print("EXPLORING SPANISH CENSUS DATA")
    print("=" * 70)
    
    # Step 1: Get the Population and Housing Censuses operation
    print("\n📊 Step 1: Finding census operations...")
    operations = api.get_operations()
    
    census_op = None
    census_op_id = None
    
    # Handle different operation formats
    if isinstance(operations, list):
        for op in operations:
            if isinstance(op, dict):
                op_id = op.get('Id') or op.get('id')
                op_name = op.get('Nombre') or op.get('nombre', '')
                if op_id == 8 or 'census' in op_name.lower() or 'población' in op_name.lower():
                    census_op = op
                    census_op_id = op_id
                    print(f"  ✓ Found: {op_name} (ID: {census_op_id})")
                    break
            elif isinstance(op, str) and 'census' in op.lower():
                print(f"  Found string operation: {op[:100]}...")
    
    if not census_op_id:
        print("  Looking for operation 8 directly...")
        census_op_id = 8
        print(f"  Using operation ID: {census_op_id}")
    
    # Step 2: Get all tables from this operation
    print(f"\n📋 Step 2: Getting tables from operation {census_op_id}...")
    tables_raw = api.get_tables_by_operation(census_op_id)
    
    print(f"  Raw response type: {type(tables_raw)}")
    
    # Process the tables based on their actual format
    tables = []
    
    if isinstance(tables_raw, list):
        for item in tables_raw:
            if isinstance(item, dict):
                tables.append(item)
            elif isinstance(item, str):
                # Try to parse string as JSON
                try:
                    parsed = json.loads(item)
                    if isinstance(parsed, dict):
                        tables.append(parsed)
                except:
                    print(f"  Skipping string that's not JSON: {item[:50]}...")
    elif isinstance(tables_raw, dict):
        # Look for arrays in the dict
        for key, value in tables_raw.items():
            if isinstance(value, list):
                tables.extend(value)
            elif isinstance(value, dict):
                tables.append(value)
    
    print(f"  Found {len(tables)} tables\n")
    
    # Display table information
    table_info = []
    for i, table in enumerate(tables, 1):
        # Try different possible field names
        table_id = table.get('Id') or table.get('id') or table.get('codigo') or f"unknown_{i}"
        table_name = table.get('Nombre') or table.get('nombre') or table.get('titulo') or 'Unknown'
        desc = table.get('Descripcion') or table.get('descripcion') or table.get('description', '')
        
        table_info.append({
            'index': i,
            'id': table_id,
            'name': table_name,
            'description': desc[:150] if desc else 'No description',
            'raw': table  # Keep raw data for debugging
        })
        
        print(f"  Table {i}:")
        print(f"    ID: {table_id}")
        print(f"    Name: {table_name}")
        if desc:
            print(f"    Description: {desc[:100]}...")
        print()
    
    # Show raw response for debugging if no tables found
    if not tables:
        print("  Raw response content:")
        print(f"  {str(tables_raw)[:500]}...")
    
    return table_info

def fetch_table_data_direct(table_id):
    """Direct fetch of table data with better error handling"""
    
    url = f"https://servicios.ine.es/wstempus/js/EN/tabla/{table_id}"
    
    print(f"\n📈 Fetching data from table {table_id}...")
    print(f"  URL: {url}")
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        data = response.json()
        
        print(f"  Response type: {type(data)}")
        
        if isinstance(data, dict):
            print(f"  Keys in response: {list(data.keys())}")
            
            # Look for data in common fields
            data_fields = ['Data', 'data', 'datos', 'observations', 'values', 'valor']
            
            for field in data_fields:
                if field in data:
                    content = data[field]
                    print(f"  Found '{field}' with {len(content) if isinstance(content, (list, dict)) else 'data'}")
                    
                    if isinstance(content, list) and len(content) > 0:
                        df = pd.DataFrame(content)
                        print(f"\n  First {min(10, len(df))} rows:")
                        print(df.head(10))
                        return df
                    elif isinstance(content, dict):
                        print(f"  Content keys: {list(content.keys())[:5]}")
                        return content
            
            # If no data field, show the structure
            print("\n  Table structure:")
            for key, value in list(data.items())[:5]:
                value_preview = str(value)[:100] if not isinstance(value, (list, dict)) else f"{type(value).__name__} with {len(value)} items"
                print(f"    {key}: {value_preview}")
            
            return data
            
        elif isinstance(data, list):
            print(f"  Retrieved list with {len(data)} items")
            if len(data) > 0:
                df = pd.DataFrame(data)
                print(f"\n  First {min(10, len(df))} rows:")
                print(df.head(10))
                return df
            else:
                print("  Empty list returned")
                return data
        else:
            print(f"  Unexpected data type: {type(data)}")
            return data
            
    except Exception as e:
        print(f"  Error: {e}")
        return None

def try_alternative_endpoints():
    """Try different API endpoints to find census data"""
    
    print("\n" + "=" * 70)
    print("TRYING ALTERNATIVE API ENDPOINTS")
    print("=" * 70)
    
    # Alternative INE API endpoints
    endpoints = [
        "https://servicios.ine.es/wstempus/js/EN/operacion/8/tablas?n=100",
        "https://servicios.ine.es/wstempus/js/EN/operacion/8",
        "https://www.ine.es/jaxiT3/DatosJson.do?c=27186",
        "https://www.ine.es/jaxiT3/Tabla.htm?t=27186",
    ]
    
    for endpoint in endpoints:
        print(f"\n📡 Trying: {endpoint}")
        try:
            response = requests.get(endpoint, timeout=10)
            print(f"  Status: {response.status_code}")
            
            if response.status_code == 200:
                content_type = response.headers.get('content-type', '')
                
                if 'json' in content_type:
                    data = response.json()
                    print(f"  Response type: {type(data)}")
                    
                    if isinstance(data, list):
                        print(f"  Retrieved {len(data)} items")
                        if len(data) > 0 and isinstance(data[0], dict):
                            print(f"  Sample keys: {list(data[0].keys())[:5]}")
                    elif isinstance(data, dict):
                        print(f"  Keys: {list(data.keys())[:5]}")
                    
                    # Try to extract a small sample
                    sample = data[:3] if isinstance(data, list) and len(data) > 3 else data
                    print(f"  Sample: {str(sample)[:200]}...")
                    
                elif 'html' in content_type:
                    print("  Received HTML (not JSON)")
                else:
                    print(f"  Content: {response.text[:200]}...")
                    
        except Exception as e:
            print(f"  Error: {e}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    
    # Step 1: Explore available census tables
    print("Step 1: Exploring census tables...")
    tables = explore_census_tables()
    
    if tables:
        print("\n" + "=" * 70)
        print("Step 2: Fetching data from tables")
        print("=" * 70)
        
        for table in tables[:3]:  # Try first 3 tables
            table_id = table['id']
            print(f"\n--- Trying table: {table['name']} (ID: {table_id}) ---")
            
            # Try to fetch data from this table
            result = fetch_table_data_direct(table_id)
            
            if result is not None and isinstance(result, pd.DataFrame) and len(result) > 0:
                # Save to CSV
                filename = f"census_table_{table_id}.csv"
                result.to_csv(filename, index=False)
                print(f"\n  ✅ Saved {len(result)} rows to {filename}")
                
                # Ask if user wants to continue
                response = input("\n  Data found! Continue exploring other tables? (y/n): ")
                if response.lower() != 'y':
                    break
    else:
        print("\n❌ No tables found through standard method.")
        print("\nTrying alternative endpoints...")
        try_alternative_endpoints()
    
    print("\n" + "=" * 70)
    print("DEBUGGING COMPLETE")
    print("=" * 70)

Step 1: Exploring census tables...
EXPLORING SPANISH CENSUS DATA

📊 Step 1: Finding census operations...
  ✓ Found: Population and Housing Censuses   (ID: 8)

📋 Step 2: Getting tables from operation 8...
  Raw response type: <class 'list'>
  Found 1 tables

  Table 1:
    ID: 8
    Name: Population and Housing Censuses  


Step 2: Fetching data from tables

--- Trying table: Population and Housing Censuses   (ID: 8) ---

📈 Fetching data from table 8...
  URL: https://servicios.ine.es/wstempus/js/EN/tabla/8
  Error: Expecting value: line 1 column 1 (char 0)

DEBUGGING COMPLETE


In [5]:
import requests
import pandas as pd
import json

class INECensusDataFinder:
    """Find and extract actual census data tables"""
    
    BASE_URL = "https://servicios.ine.es/wstempus/js/EN"
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({'Accept': 'application/json'})
    
    def get_operation_tables(self, operation_id: int):
        """Get all data tables belonging to an operation"""
        # Try different endpoints that might contain table listings
        endpoints = [
            f"{self.BASE_URL}/operacion/{operation_id}/tablas",
            f"{self.BASE_URL}/operacion/{operation_id}",
            f"{self.BASE_URL}/operacion/{operation_id}/publicaciones",
        ]
        
        for endpoint in endpoints:
            print(f"\n📡 Trying: {endpoint}")
            try:
                response = self.session.get(endpoint)
                if response.status_code == 200:
                    content_type = response.headers.get('content-type', '')
                    if 'json' in content_type:
                        data = response.json()
                        print(f"  ✓ Found JSON data of type: {type(data)}")
                        return data
                    else:
                        print(f"  Response is not JSON: {response.text[:100]}")
                else:
                    print(f"  Status: {response.status_code}")
            except Exception as e:
                print(f"  Error: {e}")
        
        return None
    
    def search_census_tables(self):
        """Search for census-related tables across all operations"""
        
        print("🔍 Searching for census data tables...")
        
        # Known census-related table IDs from INE's PC-Axis system
        # These are actual data tables from the 2021 Census
        potential_census_tables = [
            # Population tables
            {"id": "27001", "name": "Population by municipality", "type": "pc-axis"},
            {"id": "27002", "name": "Population by age groups", "type": "pc-axis"},
            {"id": "27003", "name": "Population by sex and municipality", "type": "pc-axis"},
            {"id": "27186", "name": "Population by nationality", "type": "pc-axis"},
            {"id": "27187", "name": "Household characteristics", "type": "pc-axis"},
            {"id": "27188", "name": "Housing characteristics", "type": "pc-axis"},
            {"id": "28416", "name": "Population projections", "type": "pc-axis"},
            
            # Continuous Register Statistics (Padrón)
            {"id": "32154", "name": "Official population figures", "type": "pc-axis"},
            {"id": "32155", "name": "Population by municipality (detailed)", "type": "pc-axis"},
        ]
        
        found_tables = []
        
        for table in potential_census_tables:
            print(f"\n📊 Checking table {table['id']}: {table['name']}")
            
            # Try PC-Axis endpoint
            pc_axis_url = f"https://www.ine.es/jaxiT3/DatosJson.do?c={table['id']}"
            
            try:
                response = self.session.get(pc_axis_url)
                if response.status_code == 200:
                    data = response.json()
                    if data and len(data) > 0:
                        print(f"  ✅ TABLE EXISTS with {len(data)} rows")
                        table['url'] = pc_axis_url
                        table['sample_data'] = data[:3] if len(data) > 3 else data
                        found_tables.append(table)
                    else:
                        print(f"  Table exists but has no data")
                else:
                    print(f"  Table not found or inaccessible")
            except Exception as e:
                print(f"  Error: {e}")
        
        return found_tables
    
    def fetch_pc_axis_table(self, table_id: str):
        """Fetch data from a PC-Axis table"""
        
        url = f"https://www.ine.es/jaxiT3/DatosJson.do?c={table_id}"
        
        print(f"\n📥 Fetching table {table_id}...")
        
        try:
            response = self.session.get(url)
            response.raise_for_status()
            
            data = response.json()
            
            if isinstance(data, list) and len(data) > 0:
                df = pd.DataFrame(data)
                print(f"  ✅ Retrieved {len(df)} rows, {len(df.columns)} columns")
                return df
            else:
                print(f"  Unexpected data format: {type(data)}")
                return None
                
        except Exception as e:
            print(f"  Error: {e}")
            return None

def get_population_by_municipality():
    """Specifically get population data for all municipalities"""
    
    print("\n" + "=" * 70)
    print("GETTING POPULATION BY MUNICIPALITY")
    print("=" * 70)
    
    # Method 1: Try to get from Continuous Register (Padrón)
    # The Padrón has official population figures for all municipalities
    
    endpoints_to_try = [
        # Padrón continuous statistics
        "https://www.ine.es/jaxiT3/DatosJson.do?c=32154",  # Official population figures
        "https://www.ine.es/jaxiT3/DatosJson.do?c=32155",  # Population by municipality
        "https://www.ine.es/jaxiT3/DatosJson.do?c=27186",  # Census population
    ]
    
    for endpoint in endpoints_to_try:
        print(f"\n🔍 Trying: {endpoint}")
        try:
            response = requests.get(endpoint, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if isinstance(data, list) and len(data) > 0:
                    df = pd.DataFrame(data)
                    print(f"  ✅ Success! Retrieved {len(df)} records")
                    print(f"\n  Columns: {list(df.columns)}")
                    print(f"\n  First 5 rows:")
                    print(df.head())
                    
                    # Save to CSV
                    filename = "spain_municipality_population.csv"
                    df.to_csv(filename, index=False)
                    print(f"\n  💾 Saved to {filename}")
                    return df
                else:
                    print(f"  Data format issue: {type(data)}")
            else:
                print(f"  Status: {response.status_code}")
        except Exception as e:
            print(f"  Error: {e}")
    
    print("\n  Could not retrieve municipality data directly.")
    return None

def browse_ine_census_website():
    """Provide guidance on finding actual table IDs from INE website"""
    
    print("\n" + "=" * 70)
    print("HOW TO FIND CENSUS TABLE IDS MANUALLY")
    print("=" * 70)
    
    print("""
    1. Go to the INE Census Portal:
       https://www.ine.es/dyngs/INEbase/en/operacion.htm?c=Estadistica_C&cid=1254736177000
    
    2. Click on "Population and Housing Census 2021"
    
    3. Navigate to any table you're interested in (e.g., "Population by municipality")
    
    4. Look for the table ID in:
       - The page URL: .../tabla?t=XXXXX
       - The network tab (F12) when the table loads
       - The data download link (right-click on CSV/JSON download button)
    
    5. Common census table ID ranges:
       - 27xxx: Population characteristics
       - 28xxx: Household characteristics  
       - 29xxx: Housing characteristics
       - 32xxx: Continuous Register (Padrón) statistics
    
    6. Once you have a table ID (e.g., 27186), use:
       url = f"https://www.ine.es/jaxiT3/DatosJson.do?c={table_id}"
    """)

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    
    finder = INECensusDataFinder()
    
    print("OPTION 1: Search for census tables automatically")
    print("-" * 50)
    census_tables = finder.search_census_tables()
    
    if census_tables:
        print(f"\n✅ Found {len(census_tables)} working census tables:")
        for table in census_tables:
            print(f"  • {table['name']} (ID: {table['id']})")
            
            # Fetch the first found table
            print(f"\n📊 Fetching {table['name']}...")
            df = finder.fetch_pc_axis_table(table['id'])
            if df is not None:
                df.to_csv(f"census_{table['id']}.csv", index=False)
                print(f"  Saved to census_{table['id']}.csv")
                break
    else:
        print("\n❌ No automatic tables found.")
        
        print("\nOPTION 2: Try getting municipality population data")
        print("-" * 50)
        pop_data = get_population_by_municipality()
        
        if pop_data is None:
            print("\nOPTION 3: Manual guidance")
            print("-" * 50)
            browse_ine_census_website()

OPTION 1: Search for census tables automatically
--------------------------------------------------
🔍 Searching for census data tables...

📊 Checking table 27001: Population by municipality
  Table not found or inaccessible

📊 Checking table 27002: Population by age groups
  Table not found or inaccessible

📊 Checking table 27003: Population by sex and municipality
  Table not found or inaccessible

📊 Checking table 27186: Population by nationality
  Table not found or inaccessible

📊 Checking table 27187: Household characteristics
  Table not found or inaccessible

📊 Checking table 27188: Housing characteristics
  Table not found or inaccessible

📊 Checking table 28416: Population projections
  Table not found or inaccessible

📊 Checking table 32154: Official population figures
  Table not found or inaccessible

📊 Checking table 32155: Population by municipality (detailed)
  Table not found or inaccessible

❌ No automatic tables found.

OPTION 2: Try getting municipality population da

In [6]:
import requests
import pandas as pd
import json

class INEOpenAPIClient:
    """Correct client using INE's official OpenAPI"""
    
    # Base URL from the OpenAPI documentation
    BASE_URL = "https://www.ine.es/OpenAPI"
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'Accept': 'application/json',
            'User-Agent': 'INE-OpenAPI-Client/1.0'
        })
    
    def get_operations(self, page: int = 1, limit: int = 500):
        """Get statistical operations (includes Census operations)"""
        url = f"{self.BASE_URL}/operaciones"
        params = {'page': page, 'limit': limit}
        
        response = self.session.get(url, params=params)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error {response.status_code}: {response.text}")
            return None
    
    def get_tables_by_operation(self, operation_id: int, page: int = 1):
        """Get tables belonging to a specific operation"""
        url = f"{self.BASE_URL}/operaciones/{operation_id}/tablas"
        params = {'page': page}
        
        response = self.session.get(url, params=params)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error {response.status_code}: {response.text}")
            return None
    
    def get_table_data(self, table_id: int, page: int = 1):
        """Get data from a specific table"""
        url = f"{self.BASE_URL}/tablas/{table_id}/data"
        params = {'page': page}
        
        response = self.session.get(url, params=params)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error {response.status_code}: {response.text}")
            return None
    
    def get_series_data(self, series_code: str):
        """Get data from a specific series"""
        url = f"{self.BASE_URL}/series/{series_code}/data"
        
        response = self.session.get(url)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error {response.status_code}: {response.text}")
            return None

def find_census_tables():
    """Find actual census tables using the working API"""
    
    client = INEOpenAPIClient()
    
    print("=" * 70)
    print("FINDING CENSUS TABLES VIA OFFICIAL OPENAPI")
    print("=" * 70)
    
    # Step 1: Get all operations
    print("\n📊 Step 1: Fetching operations...")
    operations = client.get_operations(page=1)
    
    if not operations:
        print("  Could not fetch operations")
        return None
    
    print(f"  Retrieved {len(operations)} operations")
    
    # Find census operations
    census_ops = []
    for op in operations:
        op_name = op.get('Nombre', '').lower()
        if any(keyword in op_name for keyword in ['censo', 'census', 'población', 'population', 'housing', 'vivienda']):
            census_ops.append(op)
            print(f"  Found: {op.get('Nombre')} (ID: {op.get('Id')})")
    
    if not census_ops:
        print("\n  No census operations found. Here are all operations:")
        for op in operations[:10]:
            print(f"    - {op.get('Id')}: {op.get('Nombre')}")
        return None
    
    # Step 2: Get tables from each census operation
    all_tables = []
    for op in census_ops:
        op_id = op.get('Id')
        op_name = op.get('Nombre')
        
        print(f"\n📋 Step 2: Getting tables from '{op_name}' (ID: {op_id})...")
        tables = client.get_tables_by_operation(op_id)
        
        if tables:
            print(f"  Found {len(tables)} tables")
            for table in tables[:10]:  # Show first 10
                table_id = table.get('Id')
                table_name = table.get('Nombre', 'Unknown')
                print(f"    - Table {table_id}: {table_name}")
                all_tables.append({
                    'operation_id': op_id,
                    'operation_name': op_name,
                    'table_id': table_id,
                    'table_name': table_name
                })
        else:
            print(f"  No tables found for this operation")
    
    return all_tables

def fetch_census_table_data():
    """Fetch actual data from a census table"""
    
    client = INEOpenAPIClient()
    
    # First, find the census tables
    tables = find_census_tables()
    
    if not tables:
        print("\n❌ No census tables found via API")
        return None
    
    print("\n" + "=" * 70)
    print("FETCHING DATA FROM CENSUS TABLES")
    print("=" * 70)
    
    # Try to fetch data from each table found
    for table in tables[:3]:  # Try first 3 tables
        table_id = table['table_id']
        table_name = table['table_name']
        
        print(f"\n📈 Fetching table {table_id}: {table_name}")
        
        data = client.get_table_data(table_id)
        
        if data:
            print(f"  Retrieved data successfully!")
            
            # Try to convert to DataFrame
            if isinstance(data, list):
                df = pd.DataFrame(data)
                print(f"  Data shape: {df.shape}")
                print(f"  Columns: {list(df.columns)}")
                print(f"\n  First 5 rows:")
                print(df.head())
                
                # Save to CSV
                filename = f"census_table_{table_id}.csv"
                df.to_csv(filename, index=False)
                print(f"\n  💾 Saved to {filename}")
                
                return df
            elif isinstance(data, dict):
                print(f"  Data keys: {list(data.keys())}")
                # Look for nested data
                for key in ['Data', 'data', 'values', 'observations']:
                    if key in data:
                        df = pd.DataFrame(data[key])
                        print(f"  Found data in '{key}' key, shape: {df.shape}")
                        return df
            else:
                print(f"  Unexpected data type: {type(data)}")
                print(f"  Sample: {str(data)[:200]}")
        else:
            print(f"  Failed to fetch data for table {table_id}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    
    print("INE CENSUS DATA RETRIEVAL VIA OFFICIAL OPENAPI")
    print("Based on documentation: https://www.ine.es/OpenAPI/index.html")
    print()
    
    # Try to fetch census data
    result = fetch_census_table_data()
    
    if result is None:
        print("\n" + "=" * 70)
        print("TROUBLESHOOTING")
        print("=" * 70)
        print("""
        If no data was retrieved, the issue may be:
        
        1. The API requires specific table IDs that we don't have yet
        2. The census data might be behind a different endpoint
        3. You may need to use the specific Census 2021 API mentioned in the OpenData portal
        
        Next steps:
        1. Visit: https://www.ine.es/OpenAPI/index.html
        2. Look for the "API JSON Censos de población y Viviendas 2021" section
        3. That API has a different base URL specifically for census data
        
        Would you like me to help you explore that specific Census 2021 API instead?
        """)

INE CENSUS DATA RETRIEVAL VIA OFFICIAL OPENAPI
Based on documentation: https://www.ine.es/OpenAPI/index.html

FINDING CENSUS TABLES VIA OFFICIAL OPENAPI

📊 Step 1: Fetching operations...
Error 404: <!DOCTYPE html>
<html lang="es">
<head>
<title>404</title>


<meta http-equiv="X-UA-Compatible" content="IE=edge">
<meta http-equiv="content-script-type" content="text/javascript" >
<meta http-equiv="content-style-type" content="text/css" >
<meta http-equiv="content-type" content="text/html; charset=ISO-8859-15" >
<link rel="shortcut icon" href="/menus/img/favicon.ico" type="image/x-icon">
<meta name="description" content="INE. Instituto Nacional de Estad&#237;stica. National Statistics Institute. Spanish Statistical Office. El INE elabora y distribuye estadisticas de Espana. Este servidor contiene: Censos de Poblacion y Viviendas 2001, Informacion general, Productos de difusion, Espana en cifras, Datos coyunturales, Datos municipales, etc.. Q2016.es">
<meta name="keywords" content="Censos p

In [7]:
!pip install censosine21

  Obtaining dependency information for censosine21 from https://files.pythonhosted.org/packages/2a/8d/6a42bd35b7c4bbd1764c391572ad6400272cc8ea37d5d4ad2cd677c58082/censosine21-0.1.0-py3-none-any.whl.metadata

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
from censosine21 import CensosINE21
import pandas as pd

# 1. Initialize the client
censo_client = CensosINE21()

# 2. Make the POST request - NOTE: positional arguments, NO keywords!
# Order: (table, metrics, variables, language)
censo_client.post("per.ppal", "SPERSONAS", ["ID_RESIDENCIA_N1", "ID_SEXO"], "EN")

# 3. The data is now stored in the client's 'data' attribute
if censo_client.data:
    # Convert to DataFrame
    df = pd.DataFrame(censo_client.data)
    
    print(f"✅ Census data retrieved successfully!")
    print(f"Total records: {len(df)}")
    print("\nFirst 5 rows:")
    print(df.head())
    
    # Save to CSV
    df.to_csv("spain_census_2021.csv", index=False)
    print("\n💾 Data saved to 'spain_census_2021.csv'")
else:
    print("❌ No data retrieved. Check your parameters.")

✅ Census data retrieved successfully!
Total records: 38

First 5 rows:
   SPERSONAS         ID_RESIDENCIA_N1 ID_SEXO
0  4182621.0                Andalucía     Man
1  4302186.0                Andalucía   Woman
2   658263.0                   Aragón     Man
3   673674.0                   Aragón   Woman
4   483186.0  Asturias, Principado de     Man

💾 Data saved to 'spain_census_2021.csv'


In [10]:
from censosine21 import CensosINE21
import pandas as pd

# Population by province and sex
censo_client = CensosINE21()
censo_client.post("per.ppal", "SPERSONAS", ["ID_RESIDENCIA_N2", "ID_SEXO"], "EN")

if censo_client.data:
    df = pd.DataFrame(censo_client.data)
    print(f"Total records: {len(df)}")
    print(df.head(10))
    df.to_csv("spain_census_by_province.csv", index=False)

Total records: 104
      ID_RESIDENCIA_N2  SPERSONAS ID_SEXO
0          02 Albacete   193488.0     Man
1          02 Albacete   193239.0   Woman
2  03 Alicante/Alacant   935232.0     Man
3  03 Alicante/Alacant   951801.0   Woman
4           04 Almería   372825.0     Man
5           04 Almería   357609.0   Woman
6       01 Araba/Álava   164298.0     Man
7       01 Araba/Álava   169188.0   Woman
8          33 Asturias   483186.0     Man
9          33 Asturias   528930.0   Woman


In [11]:
# Population by region and nationality (Spanish vs Foreigner)
censo_client = CensosINE21()
censo_client.post("per.ppal", "SPERSONAS", ["ID_RESIDENCIA_N1", "ID_NACIONALIDAD_N1"], "EN")

if censo_client.data:
    df = pd.DataFrame(censo_client.data)
    print(df.head(10))
    df.to_csv("spain_census_by_nationality.csv", index=False)

  ID_NACIONALIDAD_N1  SPERSONAS         ID_RESIDENCIA_N1
0            Spanish  7771404.0                Andalucía
1            Foreign   713400.0                Andalucía
2            Spanish  1164546.0                   Aragón
3            Foreign   167391.0                   Aragón
4            Spanish   966807.0  Asturias, Principado de
5            Foreign    45309.0  Asturias, Principado de
6            Spanish   954666.0           Balears, Illes
7            Foreign   228747.0           Balears, Illes
8            Spanish  1889940.0                 Canarias
9            Foreign   288984.0                 Canarias


In [12]:
# Number of households by region
hogares = CensosINE21()
hogares.post("hog", "SHOGARES", ["ID_RESIDENCIA_N1"], "EN")

if hogares.data:
    df = pd.DataFrame(hogares.data)
    print("Households by region:")
    print(df)
    df.to_csv("spain_households_by_region.csv", index=False)

Households by region:
     SHOGARES             ID_RESIDENCIA_N1
0   3242994.0                    Andalucía
1    540234.0                       Aragón
2    446571.0      Asturias, Principado de
3    441537.0               Balears, Illes
4    820341.0                     Canarias
5    238230.0                    Cantabria
6   1025739.0              Castilla y León
7    799722.0         Castilla - La Mancha
8   2989359.0                     Cataluña
9   2022501.0         Comunitat Valenciana
10   434364.0                  Extremadura
11  1091091.0                      Galicia
12  2546841.0         Madrid, Comunidad de
13   540663.0            Murcia, Región de
14   256329.0  Navarra, Comunidad Foral de
15   919503.0                   País Vasco
16   132240.0                    Rioja, La
17    25380.0                        Ceuta
18    25578.0                      Melilla


In [13]:
# Calculate total population by region (summing male + female)
total_by_region = df.groupby('ID_RESIDENCIA_N1')['SPERSONAS'].sum().sort_values(ascending=False)
print("\nTotal population by region:")
print(total_by_region)

# Calculate gender ratio by region
gender_ratio = df.pivot(index='ID_RESIDENCIA_N1', columns='ID_SEXO', values='SPERSONAS')
gender_ratio['Females_per_1000_Males'] = (gender_ratio['Woman'] / gender_ratio['Man']) * 1000
print("\nGender ratio (females per 1000 males):")
print(gender_ratio[['Man', 'Woman', 'Females_per_1000_Males']].round(1))

# Find regions with more females than males
regions_more_females = gender_ratio[gender_ratio['Woman'] > gender_ratio['Man']]
print("\nRegions with more women than men:")
print(regions_more_females[['Man', 'Woman']])

KeyError: 'Column not found: SPERSONAS'

In [14]:
from censosine21 import CensosINE21
import pandas as pd

# Get households by municipality (highest level available in Python)
hogares = CensosINE21()
hogares.post("hog", "SHOGARES", ["ID_RESIDENCIA_N3"], "EN")

if hogares.data:
    df = pd.DataFrame(hogares.data)
    print(f"Total records: {len(df)}")
    print(df.head(10))
    df.to_csv("spain_households_by_municipality.csv", index=False)

Total records: 8131
   SHOGARES           ID_RESIDENCIA_N3
0    2046.0  01051 Agurain/Salvatierra
1    1122.0     01001 Alegría-Dulantzi
2    4158.0              01002 Amurrio
3      81.0                01049 Añana
4     543.0              01003 Aramaio
5      96.0              01006 Armiñón
6     363.0        01037 Arraia-Maeztu
7     384.0  01008 Arratzua-Ubarrundia
8     714.0           01004 Artziniega
9     741.0            01009 Asparrena


In [15]:
import requests

# INE's official API endpoint
url = "https://www.ine.es/OpenAPI/tablas/{table_id}/data"

# You need to find the specific table_id for census tract data
# These can be found by first querying /operaciones and /tablas endpoints

response = requests.get(url)
data = response.json()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [16]:
import requests

# Example structure - you need to find the actual URLs
official_url = "https://www.ine.es/experimental/atlas/data/tract_income_2021.csv"

response = requests.get(official_url)
with open("census_tract_data.csv", "wb") as f:
    f.write(response.content)

In [17]:
from censosine21 import CensosINE21
import pandas as pd

# Get municipality-level data (8,131 municipalities)
censo_client = CensosINE21()
censo_client.post("per.ppal", "SPERSONAS", ["ID_RESIDENCIA_N3", "ID_SEXO"], "EN")

if censo_client.data:
    df = pd.DataFrame(censo_client.data)
    print(f"Retrieved {len(df)} records")
    print(df.head())
    df.to_csv("spain_census_by_municipality.csv", index=False)

Retrieved 16100 records
            ID_RESIDENCIA_N3  SPERSONAS ID_SEXO
0  01051 Agurain/Salvatierra     2523.0     Man
1  01051 Agurain/Salvatierra     2499.0   Woman
2     01001 Alegría-Dulantzi     1518.0     Man
3     01001 Alegría-Dulantzi     1407.0   Woman
4              01002 Amurrio     5085.0     Man


In [18]:
"""
Download INE Atlas de Distribución de Renta de los Hogares (ADRH)
Table: Indicadores de renta media y mediana — municipios, distritos y secciones censales
Table ID: 31097
Series: 2015–2023

Usage:
    python download_ine_adrh.py

Downloads a ~300 MB CSV file from the INE public API.
No authentication required.
"""

import requests
import os
import sys
from pathlib import Path

# --- Configuration ---
TABLE_ID = 31097
OUTPUT_DIR = Path(".")          # change to your preferred folder
FORMAT = "csv_bdsc"             # semicolon-separated CSV  (best for pandas)
                                # alternatives: csv_bd (tab), xlsx, px, json

# INE direct file URL (no JS needed, works without a browser)
URL = f"https://www.ine.es/jaxiT3/files/t/es/{FORMAT}/{TABLE_ID}.csv"

OUTPUT_FILE = OUTPUT_DIR / f"adrh_{TABLE_ID}_renta_media_mediana.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/csv,application/octet-stream,*/*",
    "Referer": "https://www.ine.es/dynt3/inebase/index.htm?padre=12385&capsel=5650",
}

def download(url: str, dest: Path) -> None:
    print(f"Downloading:\n  {url}")
    print(f"Saving to:  {dest}\n")

    with requests.get(url, headers=HEADERS, stream=True, timeout=120) as r:
        r.raise_for_status()

        total = int(r.headers.get("Content-Length", 0))
        downloaded = 0
        chunk_size = 1024 * 256   # 256 KB chunks

        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total:
                        pct = downloaded / total * 100
                        mb = downloaded / 1e6
                        sys.stdout.write(f"\r  {mb:.1f} MB / {total/1e6:.1f} MB  ({pct:.1f}%)")
                    else:
                        sys.stdout.write(f"\r  {downloaded/1e6:.1f} MB downloaded")
                    sys.stdout.flush()

    print(f"\n\nDone! File saved: {dest}  ({os.path.getsize(dest)/1e6:.1f} MB)")


if __name__ == "__main__":
    download(URL, OUTPUT_FILE)

    # Quick preview with pandas (optional)
    try:
        import pandas as pd
        print("\nPreviewing first 5 rows:")
        df = pd.read_csv(OUTPUT_FILE, sep=";", encoding="utf-8-sig", nrows=5)
        print(df.to_string())
    except ImportError:
        print("\n(Install pandas to get a quick preview: pip install pandas)")

Downloading:
  https://www.ine.es/jaxiT3/files/t/es/csv_bdsc/31097.csv
Saving to:  adrh_31097_renta_media_mediana.csv

  33.4 MB downloaded

Done! File saved: adrh_31097_renta_media_mediana.csv  (33.4 MB)

Previewing first 5 rows:
          Municipios  Distritos  Secciones Indicadores de renta media y mediana  Periodo   Total
0  28001 Acebeda, La        NaN        NaN         Renta neta media por persona     2023  16.893
1  28001 Acebeda, La        NaN        NaN         Renta neta media por persona     2022  15.715
2  28001 Acebeda, La        NaN        NaN         Renta neta media por persona     2021  15.245
3  28001 Acebeda, La        NaN        NaN         Renta neta media por persona     2020  13.999
4  28001 Acebeda, La        NaN        NaN         Renta neta media por persona     2019       .


In [19]:
"""
Filter ADRH table 31097 to sección censal level only and pivot to wide format.
Output: one row per (sección censal × year), one column per indicator.
"""

import pandas as pd

INPUT  = "adrh_31097_renta_media_mediana.csv"
OUTPUT = "adrh_secciones_renta.parquet"   # parquet keeps dtypes and is much smaller

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(INPUT, sep=";", encoding="utf-8-sig")
df.columns = ["municipio", "distrito", "seccion", "indicador", "periodo", "valor"]

print(f"Rows loaded: {len(df):,}")

# ── Keep only sección censal rows (seccion column is NOT null) ─────────────────
secciones = df[df["seccion"].notna()].copy()
print(f"Rows at sección level: {len(secciones):,}")
print(f"Unique secciones: {secciones['seccion'].nunique():,}")
print(f"Years: {sorted(secciones['periodo'].unique())}")
print(f"Indicators:\n  " + "\n  ".join(sorted(secciones["indicador"].unique())))

# ── Clean values  ('.' → NaN, European decimal comma → dot) ──────────────────
secciones["valor"] = (
    secciones["valor"]
    .astype(str)
    .str.strip()
    .replace(".", None)                          # suppressed cells
    .str.replace(".", "", regex=False)           # thousands separator
    .str.replace(",", ".", regex=False)          # decimal comma
    .pipe(pd.to_numeric, errors="coerce")
)

# ── Extract clean sección code (10-digit string: 2 prov + 3 mun + 2 dist + 3 sec) ──
# The seccion column looks like "2800101001 Acebeda, La sección 01001"
secciones["cod_seccion"] = secciones["seccion"].str.extract(r"^(\d{10})")

# Also extract province and municipality codes for easy grouping
secciones["cod_provincia"] = secciones["cod_seccion"].str[:2]
secciones["cod_municipio"] = secciones["cod_seccion"].str[:5]

# ── Pivot: wide format, one row per sección × year ───────────────────────────
wide = (
    secciones
    .pivot_table(
        index=["cod_seccion", "cod_provincia", "cod_municipio", "municipio", "periodo"],
        columns="indicador",
        values="valor",
        aggfunc="first",
    )
    .reset_index()
)

# Flatten column names
wide.columns.name = None

# Friendly short column names
rename = {
    "Renta neta media por persona":       "renta_neta_media_persona",
    "Renta neta media por hogar":         "renta_neta_media_hogar",
    "Media de la renta por unidad de consumo":   "renta_media_uc",
    "Mediana de la renta por unidad de consumo": "renta_mediana_uc",
    "Renta bruta media por persona":      "renta_bruta_media_persona",
    "Renta bruta media por hogar":        "renta_bruta_media_hogar",
}
wide.rename(columns=rename, inplace=True)

print(f"\nFinal shape: {wide.shape}")
print(wide.head(10).to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
wide.to_parquet(OUTPUT, index=False)
print(f"\nSaved: {OUTPUT}  ({wide.memory_usage(deep=True).sum()/1e6:.1f} MB in memory)")

# Also save CSV if needed
wide.to_csv(OUTPUT.replace(".parquet", ".csv"), index=False)
print(f"Saved: {OUTPUT.replace('.parquet', '.csv')}")

/tmp/ipykernel_13483/527169449.py:12: DtypeWarning: Columns (0: Total) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT, sep=";", encoding="utf-8-sig")


Rows loaded: 268,164
Rows at sección level: 245,214
Unique secciones: 4,541
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Indicators:
  Media de la renta por unidad de consumo
  Mediana de la renta por unidad de consumo
  Renta bruta media por hogar
  Renta bruta media por persona
  Renta neta media por hogar
  Renta neta media por persona

Final shape: (39689, 11)
  cod_seccion cod_provincia cod_municipio          municipio  periodo  renta_media_uc  renta_mediana_uc  renta_bruta_media_hogar  renta_bruta_media_persona  renta_neta_media_hogar  renta_neta_media_persona
0  2800101001            28         28001  28001 Acebeda, La     2020             NaN               NaN                  32211.0                    16452.0                 27408.0                   13999.0
1  2800101001            28         28001  28001 Acebeda, La     2021             NaN               NaN           

In [21]:
"""
Download and process the Censo de Población y Viviendas 2021
Indicator file for secciones censales — single file, all Spain.

Source: https://www.ine.es/dyngs/INEbase/operacion.htm?c=Estadistica_C&cid=1254736177108&menu=resultados
Direct: https://www.ine.es/censos2021/C2021_Indicadores.csv

Keeps only the indicators relevant for HVAC and solar propensity models:
  - Ownership rate (% viviendas en propiedad)
  - Building age distribution
  - Building type (apartment vs house)
  - Household size
  - Population structure
"""

import requests
import pandas as pd
from pathlib import Path
import sys

OUTPUT_DIR = Path(".")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Referer": "https://www.ine.es/",
}

# ── Download ──────────────────────────────────────────────────────────────────
def download(url: str, dest: Path) -> Path:
    if dest.exists():
        print(f"Already downloaded: {dest}")
        return dest

    print(f"Downloading: {url}")
    with requests.get(url, headers=HEADERS, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))
        downloaded = 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=512 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    pct = f" ({downloaded/total*100:.0f}%)" if total else ""
                    sys.stdout.write(f"\r  {downloaded/1e6:.1f} MB{pct}")
                    sys.stdout.flush()
    print(f"\n  Saved: {dest}  ({dest.stat().st_size/1e6:.1f} MB)")
    return dest


# ── Load & inspect ────────────────────────────────────────────────────────────
def load_censo(path: Path) -> pd.DataFrame:
    # The file uses semicolon separator and has a header row + data
    df = pd.read_csv(path, sep=";", encoding="utf-8-sig", dtype=str)
    df.columns = [c.strip() for c in df.columns]
    print(f"\nLoaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"First columns: {list(df.columns[:8])}")
    print(f"\nAll column names:")
    for i, c in enumerate(df.columns):
        print(f"  [{i:03d}] {c}")
    return df


# ── Select relevant indicators ────────────────────────────────────────────────
# These are the known column name patterns in C2021_Indicadores.csv
# Exact names confirmed from INE documentation

INDICATOR_MAP = {
    # Geography
    "CUSEC":            "cod_seccion",       # 10-digit section code (key join field)
    "NMUN":             "municipio",
    "NPRO":             "provincia",
    "NCA":              "ccaa",

    # Population
    "t1_1":             "poblacion_total",
    "t1_2":             "poblacion_hombres",
    "t1_3":             "poblacion_mujeres",

    # Age structure
    "t2_1":             "pct_menores_16",
    "t2_2":             "pct_16_64",
    "t2_3":             "pct_65_mas",
    "t2_4":             "edad_media",

    # Household
    "t6_1":             "n_hogares",
    "t6_2":             "tamano_medio_hogar",
    "t6_3":             "pct_hogares_unipersonales",

    # Housing tenure — KEY for both solar and HVAC
    "t8_1":             "n_viviendas_principales",
    "t8_2":             "pct_vivienda_propiedad",       # % owned (not renting)
    "t8_3":             "pct_vivienda_alquiler",        # % renting
    "t8_4":             "pct_vivienda_otra_forma",      # % other tenure

    # Building type — KEY for solar (houses > flats) and HVAC sizing
    "t9_1":             "pct_edificio_unifamiliar",     # detached/semi-detached
    "t9_2":             "pct_edificio_bloque",          # apartment block

    # Building age — KEY for replacement demand (old = no system or outdated)
    "t10_1":            "pct_antes_1900",
    "t10_2":            "pct_1900_1940",
    "t10_3":            "pct_1941_1960",
    "t10_4":            "pct_1961_1970",
    "t10_5":            "pct_1971_1980",
    "t10_6":            "pct_1981_1990",
    "t10_7":            "pct_1991_2000",
    "t10_8":            "pct_2001_2010",
    "t10_9":            "pct_despues_2010",

    # Housing surface — proxy for ticket size
    "t11_1":            "pct_menos_30m2",
    "t11_2":            "pct_30_45m2",
    "t11_3":            "pct_45_60m2",
    "t11_4":            "pct_60_75m2",
    "t11_5":            "pct_75_90m2",
    "t11_6":            "pct_90_105m2",
    "t11_7":            "pct_105_120m2",
    "t11_8":            "pct_mas_120m2",

    # Education (proxy for income stability)
    "t4_1":             "pct_sin_estudios",
    "t4_2":             "pct_estudios_primarios",
    "t4_3":             "pct_estudios_secundarios",
    "t4_4":             "pct_estudios_superiores",

    # Employment
    "t5_1":             "pct_ocupados",
    "t5_2":             "pct_parados",
    "t5_3":             "pct_inactivos",

    # Nationality (affects income attribution in ADRH)
    "t3_1":             "pct_espanoles",
    "t3_2":             "pct_extranjeros",
}


def select_and_rename(df: pd.DataFrame) -> pd.DataFrame:
    # Find which expected columns actually exist
    available  = {k: v for k, v in INDICATOR_MAP.items() if k in df.columns}
    missing    = [k for k in INDICATOR_MAP if k not in df.columns]

    if missing:
        print(f"\nWarning: {len(missing)} expected columns not found (may have different names):")
        print(f"  {missing}")
        print("\nSearching for similar columns...")
        # Try to find close matches
        for m in missing[:5]:
            close = [c for c in df.columns if m.lower() in c.lower() or c.lower().startswith(m[:3].lower())]
            if close:
                print(f"  '{m}' → possible matches: {close[:3]}")

    selected = df[list(available.keys())].copy()
    selected.rename(columns=available, inplace=True)

    # Convert numeric columns
    for col in selected.columns:
        if col not in ["cod_seccion", "municipio", "provincia", "ccaa"]:
            selected[col] = (
                selected[col]
                .str.replace(",", ".", regex=False)
                .str.replace(" ", "", regex=False)
                .pipe(pd.to_numeric, errors="coerce")
            )

    # Derived features useful for the model
    if "pct_1941_1960" in selected.columns and "pct_1961_1970" in selected.columns:
        selected["pct_edificios_pre1980"] = (
            selected[["pct_antes_1900","pct_1900_1940","pct_1941_1960",
                       "pct_1961_1970","pct_1971_1980"]]
            .sum(axis=1, skipna=True)
        )

    return selected


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Download
    raw = download(
        "https://www.ine.es/censos2021/C2021_Indicadores.csv",
        OUTPUT_DIR / "censo2021_indicadores_raw.csv",
    )

    # 2. Load and inspect all columns
    df = load_censo(raw)

    # 3. Select relevant indicators
    selected = select_and_rename(df)
    print(f"\nSelected shape: {selected.shape}")
    print(f"Columns: {list(selected.columns)}")
    print(selected.head(3).to_string())

    # 4. Save
    selected.to_parquet(OUTPUT_DIR / "censo2021_secciones.parquet", index=False)
    selected.to_csv(OUTPUT_DIR / "censo2021_secciones.csv", index=False)
    print(f"\nSaved: censo2021_secciones.parquet  ({selected.memory_usage(deep=True).sum()/1e6:.1f} MB)")
    print(f"Saved: censo2021_secciones.csv")

    # 5. Merge with ADRH 2023 if available
    adrh_path = OUTPUT_DIR / "adrh_secciones_2023.parquet"
    if adrh_path.exists():
        print("\nMerging with ADRH 2023...")
        adrh  = pd.read_parquet(adrh_path)
        # Align keys: ADRH cod_seccion is already 10-digit string
        merged = adrh.merge(
            selected.rename(columns={"cod_seccion": "cod_seccion"}),
            on="cod_seccion",
            how="left",
            suffixes=("_adrh", "_censo"),
        )
        merged.to_parquet(OUTPUT_DIR / "features_secciones_2023.parquet", index=False)
        merged.to_csv(OUTPUT_DIR / "features_secciones_2023.csv", index=False)
        print(f"Merged shape: {merged.shape}")
        print(f"Saved: features_secciones_2023.parquet  ← use this as model input")
    else:
        print("\nNote: run adrh_secciones.py first to generate adrh_secciones_2023.parquet")
        print("Then re-run this script for the auto-merge.")

Already downloaded: censo2021_indicadores_raw.csv

Loaded: 36,333 rows × 1 columns
First columns: ['ccaa,cpro,cmun,dist,secc,t1_1,t2_1,t2_2,t3_1,t4_1,t4_2,t4_3,t5_1,t6_1,t7_1,t8_1,t9_1,t10_1,t11_1,t12_1,t13_1,t14_1,t15_1,t16_1,t17_1,t17_2,t17_3,t17_4,t17_5,t18_1,t19_1,t19_2,t20_1,t20_2,t20_3,t21_1,t22_1,t22_2,t22_3,t22_4,t22_5']

All column names:
  [000] ccaa,cpro,cmun,dist,secc,t1_1,t2_1,t2_2,t3_1,t4_1,t4_2,t4_3,t5_1,t6_1,t7_1,t8_1,t9_1,t10_1,t11_1,t12_1,t13_1,t14_1,t15_1,t16_1,t17_1,t17_2,t17_3,t17_4,t17_5,t18_1,t19_1,t19_2,t20_1,t20_2,t20_3,t21_1,t22_1,t22_2,t22_3,t22_4,t22_5

  ['CUSEC', 'NMUN', 'NPRO', 'NCA', 't1_1', 't1_2', 't1_3', 't2_1', 't2_2', 't2_3', 't2_4', 't6_1', 't6_2', 't6_3', 't8_1', 't8_2', 't8_3', 't8_4', 't9_1', 't9_2', 't10_1', 't10_2', 't10_3', 't10_4', 't10_5', 't10_6', 't10_7', 't10_8', 't10_9', 't11_1', 't11_2', 't11_3', 't11_4', 't11_5', 't11_6', 't11_7', 't11_8', 't4_1', 't4_2', 't4_3', 't4_4', 't5_1', 't5_2', 't5_3', 't3_1', 't3_2']

Searching for similar c

In [22]:
"""
Fix and process the Censo 2021 secciones censales indicator file.

Issues fixed:
  1. Separator is comma, not semicolon
  2. Geography must be built from component columns: cpro + cmun + dist + secc
  3. Column names are t-codes — read the indicator dictionary to map them

Run this ONCE first to inspect columns and values:
    python censo2021_fix.py --inspect

Then run normally to process:
    python censo2021_fix.py
"""

import pandas as pd
from pathlib import Path
import sys

RAW_CSV   = Path("censo2021_indicadores_raw.csv")
DICT_XLSX = Path("censo2021_indicadores_dict.xlsx")   # download from INE if available
OUTPUT_DIR = Path(".")

# ── Step 1: Read with correct separator ──────────────────────────────────────
def load_raw() -> pd.DataFrame:
    df = pd.read_csv(RAW_CSV, sep=",", encoding="utf-8-sig", dtype=str)
    df.columns = [c.strip() for c in df.columns]
    print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
    return df


# ── Step 2: Print column sample so you can see what each t-code contains ─────
def inspect(df: pd.DataFrame):
    print("\n=== COLUMN INSPECTION ===")
    print(f"{'Column':<12} {'Sample value 1':<20} {'Sample value 2':<20} {'Sample value 3'}")
    print("-" * 80)
    for col in df.columns:
        samples = df[col].dropna().head(3).tolist()
        s1 = str(samples[0])[:18] if len(samples) > 0 else ""
        s2 = str(samples[1])[:18] if len(samples) > 1 else ""
        s3 = str(samples[2])[:18] if len(samples) > 2 else ""
        print(f"{col:<12} {s1:<20} {s2:<20} {s3}")


# ── Step 3: Try to read the indicator dictionary XLSX ────────────────────────
def load_dict() -> dict:
    """
    The INE indicator dictionary XLSX maps t-codes to descriptions.
    Download it from:
    https://www.ine.es/censos2021/indicadores_seccen_c2021.xlsx
    Save as: censo2021_indicadores_dict.xlsx
    """
    if not DICT_XLSX.exists():
        print(f"\nWarning: {DICT_XLSX} not found.")
        print("Download it from: https://www.ine.es/censos2021/indicadores_seccen_c2021.xlsx")
        print("Save as: censo2021_indicadores_dict.xlsx")
        print("Then re-run. Proceeding with best-guess mapping.\n")
        return {}

    try:
        ddf = pd.read_excel(DICT_XLSX, dtype=str)
        print(f"\nDictionary loaded: {ddf.shape}")
        print(ddf.head(10).to_string())
        # Try to build code→description map
        # Typical structure: first col = code, second = description
        code_col = ddf.columns[0]
        desc_col = ddf.columns[1] if len(ddf.columns) > 1 else None
        if desc_col:
            mapping = dict(zip(ddf[code_col].str.strip(), ddf[desc_col].str.strip()))
            return mapping
    except Exception as e:
        print(f"Could not read dictionary: {e}")
    return {}


# ── Step 4: Known t-code mapping (best guess from INE documentation) ─────────
# Based on the Censo 2021 indicator file structure.
# Run --inspect first to verify these match your actual data values.

TCODE_MAP = {
    # Geography (component columns — will be combined into cod_seccion)
    "ccaa":   "ccaa",
    "cpro":   "cpro",
    "cmun":   "cmun",
    "dist":   "dist",
    "secc":   "secc",

    # Population
    "t1_1":   "poblacion_total",

    # Age
    "t2_1":   "edad_media",
    "t2_2":   "pct_menores_18",

    # Nationality
    "t3_1":   "pct_extranjeros",

    # Education
    "t4_1":   "pct_sin_estudios",
    "t4_2":   "pct_estudios_primarios",
    "t4_3":   "pct_estudios_superiores",

    # Activity / Employment
    "t5_1":   "tasa_paro",

    # Households
    "t6_1":   "n_hogares",

    # Household size
    "t7_1":   "tamano_medio_hogar",

    # Housing tenure — KEY for solar & HVAC
    "t8_1":   "pct_vivienda_propiedad",

    # Housing surface (m²)
    "t9_1":   "superficie_media_m2",

    # Building age — KEY for replacement demand
    "t10_1":  "pct_edificios_anterior_1900",   # verify with --inspect

    # Other housing/building indicators (verify with --inspect)
    "t11_1":  "t11_1",
    "t12_1":  "t12_1",
    "t13_1":  "t13_1",
    "t14_1":  "t14_1",
    "t15_1":  "t15_1",
    "t16_1":  "t16_1",

    # Building type breakdown
    "t17_1":  "t17_1",
    "t17_2":  "t17_2",
    "t17_3":  "t17_3",
    "t17_4":  "t17_4",
    "t17_5":  "t17_5",

    "t18_1":  "t18_1",
    "t19_1":  "t19_1",
    "t19_2":  "t19_2",
    "t20_1":  "t20_1",
    "t20_2":  "t20_2",
    "t20_3":  "t20_3",
    "t21_1":  "t21_1",
    "t22_1":  "t22_1",
    "t22_2":  "t22_2",
    "t22_3":  "t22_3",
    "t22_4":  "t22_4",
    "t22_5":  "t22_5",
}


# ── Step 5: Process ───────────────────────────────────────────────────────────
def process(df: pd.DataFrame, dict_map: dict) -> pd.DataFrame:
    # Build 10-digit sección censal code: cpro(2) + cmun(3) + dist(2) + secc(3)
    df["cod_seccion"] = (
        df["cpro"].str.zfill(2) +
        df["cmun"].str.zfill(3) +
        df["dist"].str.zfill(2) +
        df["secc"].str.zfill(3)
    )
    df["cod_provincia"] = df["cpro"].str.zfill(2)
    df["cod_municipio"] = df["cpro"].str.zfill(2) + df["cmun"].str.zfill(3)

    # Rename t-codes using dict first, then fallback to TCODE_MAP
    rename = {}
    for col in df.columns:
        if col in dict_map:
            rename[col] = dict_map[col].lower().replace(" ", "_")[:40]
        elif col in TCODE_MAP:
            rename[col] = TCODE_MAP[col]

    df = df.rename(columns=rename)

    # Convert all t-code columns to numeric
    skip = {"ccaa", "cpro", "cmun", "dist", "secc", "cod_seccion", "cod_provincia", "cod_municipio"}
    for col in df.columns:
        if col not in skip:
            df[col] = (
                df[col].astype(str).str.strip()
                .str.replace(",", ".", regex=False)
                .pipe(pd.to_numeric, errors="coerce")
            )

    # Reorder: geography first
    geo_cols   = ["cod_seccion", "cod_provincia", "cod_municipio", "ccaa", "cpro", "cmun", "dist", "secc"]
    other_cols = [c for c in df.columns if c not in geo_cols]
    df = df[geo_cols + other_cols]

    print(f"\nProcessed shape: {df.shape}")
    print(df.head(3).to_string())
    return df


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    inspect_only = "--inspect" in sys.argv

    df = load_raw()

    if inspect_only:
        inspect(df)
        print("\nRe-run without --inspect to process and save.")
        sys.exit(0)

    # Always show inspect output first
    inspect(df)

    # Load dictionary if available
    dict_map = load_dict()

    # Process
    out = process(df, dict_map)

    # Save
    out.to_parquet(OUTPUT_DIR / "censo2021_secciones.parquet", index=False)
    out.to_csv(OUTPUT_DIR / "censo2021_secciones.csv", index=False)
    print(f"\nSaved: censo2021_secciones.parquet")
    print(f"Saved: censo2021_secciones.csv")

    # Auto-merge with ADRH 2023
    adrh_path = OUTPUT_DIR / "adrh_secciones_2023.parquet"
    if adrh_path.exists():
        print("\nMerging with ADRH 2023...")
        adrh   = pd.read_parquet(adrh_path)
        merged = adrh.merge(out, on="cod_seccion", how="left")
        merged.to_parquet(OUTPUT_DIR / "features_secciones_2023.parquet", index=False)
        merged.to_csv(OUTPUT_DIR / "features_secciones_2023.csv", index=False)
        print(f"Merged shape: {merged.shape}")
        print(f"Saved: features_secciones_2023.parquet  ← use this as model input")
    else:
        print("\nNote: generate adrh_secciones_2023.parquet first for auto-merge.")

Loaded: 36,333 rows × 41 columns

=== COLUMN INSPECTION ===
Column       Sample value 1       Sample value 2       Sample value 3
--------------------------------------------------------------------------------
ccaa         01                   01                   01
cpro         04                   04                   04
cmun         001                  002                  003
dist         01                   01                   01
secc         001                  001                  001
t1_1         1260                 1212                 869
t2_1         0.4857               0.4719               0.5086
t2_2         0.5143               0.5281               0.4914
t3_1         49.4994              49.4636              45.0789
t4_1         0.1048               0.0891               0.1208
t4_2         0.6008               0.6287               0.6789
t4_3         0.2944               0.2822               0.2002
t5_1         0.0984               0.0941               0.1231
t6_